# 🍳 Optimizer Cookbook — Chapter 06: RAdam

**Previous:** `05_adamw.ipynb` — AdamW  
**Next:** `07_comparison_all.ipynb` — Final Head-to-Head

---

## The Problem: Adam's Unstable Warmup

Adam and AdamW both initialise their second moment estimate $v_0 = 0$.  
In the first few hundred steps, $v_t$ is still very small — meaning:

$$\frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \gg \eta$$

The effective learning rate is **much larger than intended** early in training, causing erratic, large updates before the model has stabilised. This is why practitioners typically add a **linear warmup schedule** — manually ramping up the LR over the first N steps before it decays.

RAdam removes the need for this manual warmup entirely.

---

## What is RAdam?

**RAdam** (Rectified Adam, Liu et al., *On the Variance of the Adaptive Learning Rate and Beyond*, 2020) introduces a **variance rectification term** $r_t$ that measures how reliable the second moment estimate is at each timestep.

- When $v_t$ is not yet reliable (early training), RAdam **falls back to SGD** — taking safe, unscaled steps
- Once $v_t$ is stable, RAdam **switches to full Adam** — using adaptive scaling

This is automatic — no warmup schedule needed.

---

## Update Rule

**Step 1 — Standard moment updates (same as Adam):**
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\nabla_t \mathcal{L}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)(\nabla_t \mathcal{L})^2$$

**Step 2 — Compute the maximum length of the approximated SMA:**
$$\rho_{\infty} = \frac{2}{1 - \beta_2} - 1$$

**Step 3 — Compute current SMA length:**
$$\rho_t = \rho_{\infty} - \frac{2t \cdot \beta_2^t}{1 - \beta_2^t}$$

**Step 4 — Conditional update:**

If $\rho_t > 4$ (variance estimate is tractable):
$$r_t = \sqrt{\frac{(\rho_t - 4)(\rho_t - 2)\rho_{\infty}}{(\rho_{\infty} - 4)(\rho_{\infty} - 2)\rho_t}}$$
$$\theta_{t+1} = \theta_t - \alpha_t \cdot r_t \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

Else (variance estimate unreliable — SGD fallback):
$$\theta_{t+1} = \theta_t - \alpha_t \cdot \hat{m}_t$$

---

## RAdam vs Adam — Key Differences

| Property | Adam | RAdam |
|---|---|---|
| Early training stability | ❌ Variance not reliable | ✅ Falls back to SGD |
| Warmup schedule required | ⚠️ Often recommended | ✅ Built-in — not needed |
| Hyperparameter sensitivity | Moderate | Lower |
| Convergence speed (later epochs) | Same | Same |
| PyTorch support | `torch.optim.Adam` | `torch.optim.RAdam` |

---

## When to Use RAdam

| Scenario | Verdict |
|---|---|
| Training from scratch without a LR scheduler | ✅ Excellent — no warmup needed |
| Tasks where Adam is sensitive to LR | ✅ RAdam is more robust |
| NLP tasks, Transformers | ✅ Good — though AdamW + warmup is still popular |
| When you want fewer hyperparameters | ✅ No warmup steps to tune |
| Quick baseline runs | ✅ Reliable out of the box |
| When you already have a warmup schedule | ➡️ Adam/AdamW are fine |

---

## Key Hyperparameters

| Parameter | Default | Notes |
|---|---|---|
| `lr` | 1e-3 | Same as Adam — RAdam is less sensitive |
| `betas` | (0.9, 0.999) | Same as Adam |
| `eps` | 1e-8 | Same as Adam |
| `weight_decay` | 0 | Use decoupled WD for regularization (combine with AdamW ideas) |

---
## 0. Imports & CUDA Check

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {device}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')

---
## 1. Config

In [ ]:
BATCH_SIZE     = 128
NUM_EPOCHS     = 20
NUM_CLASSES    = 10
NUM_WORKERS    = 2
SEED           = 42

# ── Optimizer Config ─────────────────────────────────────────
LEARNING_RATE  = 1e-3
BETA1          = 0.9
BETA2          = 0.999
EPS            = 1e-8
WEIGHT_DECAY   = 0
OPTIMIZER_NAME = 'RAdam'

DATA_DIR    = '../data'
RESULTS_DIR = '../results/logs'
PLOTS_DIR   = '../results/plots'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Config loaded ✓')

---
## 2. Dataset — CIFAR-10

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

---
## 3. Model — SimpleCNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(SEED)
model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready. Trainable params: {total_params:,}')

---
## 4. Optimizer — RAdam

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.RAdam(
    model.parameters(),
    lr           = LEARNING_RATE,
    betas        = (BETA1, BETA2),
    eps          = EPS,
    weight_decay = WEIGHT_DECAY
)

print(f'Optimizer    : RAdam')
print(f'LR           : {LEARNING_RATE}')
print(f'Betas        : β₁={BETA1}, β₂={BETA2}')
print(f'Eps          : {EPS}')
print(f'Weight Decay : {WEIGHT_DECAY}')
print('-' * 65)

---
## 5. Training Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

def run_training(model, train_loader, test_loader, optimizer, criterion, num_epochs, device, optimizer_name):
    history = []
    best_acc = 0.0
    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc     = evaluate(model, test_loader, criterion, device)
        elapsed = time.time() - t0
        if val_acc > best_acc: best_acc = val_acc
        history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                        'val_loss': val_loss, 'val_acc': val_acc, 'time_s': elapsed})
        print(f'Epoch [{epoch:02d}/{num_epochs}] Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | Time: {elapsed:.1f}s')
    print(f'\n✓ Best Val Accuracy: {best_acc:.2f}%')
    df = pd.DataFrame(history)
    df.to_csv(f'{RESULTS_DIR}/{optimizer_name}_log.csv', index=False)
    print(f'✓ Log saved → results/logs/{optimizer_name}_log.csv')
    return df

print('Utilities loaded ✓')

---
## 6. Train

In [ ]:
history = run_training(
    model          = model,
    train_loader   = train_loader,
    test_loader    = test_loader,
    optimizer      = optimizer,
    criterion      = criterion,
    num_epochs     = NUM_EPOCHS,
    device         = device,
    optimizer_name = OPTIMIZER_NAME
)

---
## 7. Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Training Curves — {OPTIMIZER_NAME}', fontsize=14, fontweight='bold')

ax1.plot(history['epoch'], history['train_loss'], label='Train Loss', linewidth=2, color='steelblue')
ax1.plot(history['epoch'], history['val_loss'],   label='Val Loss',   linewidth=2, color='tomato', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['epoch'], history['train_acc'], label='Train Acc', linewidth=2, color='steelblue')
ax2.plot(history['epoch'], history['val_acc'],   label='Val Acc',   linewidth=2, color='tomato', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/{OPTIMIZER_NAME}_curves.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. Visualising the Rectification Term $r_t$

This cell analytically computes the RAdam rectification term $r_t$ over the first 200 steps.  
It shows exactly when RAdam switches from SGD-mode to Adam-mode.

- When $\rho_t \leq 4$: SGD-like update (no adaptive scaling)
- When $\rho_t > 4$: Adam-like update with rectification factor $r_t$

In [ ]:
beta2      = BETA2
rho_inf    = 2 / (1 - beta2) - 1
steps      = 200

rho_t_vals = []
r_t_vals   = []
mode       = []   # 'sgd' or 'adam'

for t in range(1, steps + 1):
    rho_t = rho_inf - 2 * t * (beta2 ** t) / (1 - beta2 ** t)
    rho_t_vals.append(rho_t)
    if rho_t > 4:
        r_t = np.sqrt(
            ((rho_t - 4) * (rho_t - 2) * rho_inf) /
            ((rho_inf - 4) * (rho_inf - 2) * rho_t)
        )
        r_t_vals.append(r_t)
        mode.append('adam')
    else:
        r_t_vals.append(None)
        mode.append('sgd')

# Find switchover step
switchover = next((i+1 for i, m in enumerate(mode) if m == 'adam'), steps)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'RAdam — Rectification Term Analysis (β₂={beta2})', fontsize=13, fontweight='bold')

# Left: ρ_t over steps
ax1.plot(range(1, steps+1), rho_t_vals, color='steelblue', linewidth=2)
ax1.axhline(y=4, color='tomato', linestyle='--', linewidth=1.5, label='Threshold ρ=4')
ax1.axvline(x=switchover, color='gray', linestyle=':', linewidth=1.5, label=f'Switchover step={switchover}')
ax1.fill_between(range(1, steps+1), rho_t_vals, 4,
                 where=[r < 4 for r in rho_t_vals], alpha=0.15, color='tomato', label='SGD mode')
ax1.set_title('SMA Length $\\rho_t$ over Steps'); ax1.set_xlabel('Step'); ax1.set_ylabel('$\\rho_t$')
ax1.legend(); ax1.grid(alpha=0.3)

# Right: r_t rectification factor (only where Adam mode is active)
adam_steps = [i+1 for i, m in enumerate(mode) if m == 'adam']
adam_r     = [r for r, m in zip(r_t_vals, mode) if m == 'adam']
ax2.plot(adam_steps, adam_r, color='mediumseagreen', linewidth=2)
ax2.axhline(y=1.0, color='gray', linestyle='--', linewidth=1.5, label='r=1 (full Adam)')
ax2.set_title('Rectification Factor $r_t$ (Adam mode only)')
ax2.set_xlabel('Step'); ax2.set_ylabel('$r_t$')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/RAdam_rectification_term.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'ρ_∞ = {rho_inf:.1f}')
print(f'RAdam switches from SGD → Adam mode at step {switchover}')
print(f'r_t approaches 1.0 asymptotically — full Adam behaviour at convergence')

---
## 9. RAdam vs Adam — Early Epoch Comparison

RAdam's main advantage is stable early training. This cell zooms into the **first 5 epochs**  
to show whether the variance correction makes a measurable difference.

In [ ]:
EARLY_EPOCHS = 5
early_results = {}

for name, opt_class in [('Adam', optim.Adam), ('RAdam', optim.RAdam)]:
    torch.manual_seed(SEED)
    m    = SimpleCNN(num_classes=NUM_CLASSES).to(device)
    opt  = opt_class(m.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2), eps=EPS)
    crit = nn.CrossEntropyLoss()
    losses, accs = [], []
    for epoch in range(1, EARLY_EPOCHS + 1):
        tr_loss, _  = train_one_epoch(m, train_loader, crit, opt, device)
        vl_loss, vl_acc = evaluate(m, test_loader, crit, device)
        losses.append(vl_loss)
        accs.append(vl_acc)
    early_results[name] = {'loss': losses, 'acc': accs}
    print(f'{name:<10} → Epoch 1 Val Acc: {accs[0]:.2f}% | Epoch {EARLY_EPOCHS} Val Acc: {accs[-1]:.2f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Adam vs RAdam — Early Epoch Stability (Epochs 1–5)', fontsize=13, fontweight='bold')
colors = {'Adam': 'tomato', 'RAdam': 'steelblue'}

for name, res in early_results.items():
    ax1.plot(range(1, EARLY_EPOCHS+1), res['loss'], marker='o', linewidth=2, label=name, color=colors[name])
    ax2.plot(range(1, EARLY_EPOCHS+1), res['acc'],  marker='o', linewidth=2, label=name, color=colors[name])

ax1.set_title('Val Loss (early)'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_title('Val Accuracy (early)'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/RAdam_vs_Adam_early.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 10. LR Robustness — RAdam vs Adam

RAdam is claimed to be less sensitive to LR than Adam, especially at higher rates.  
This cell tests that claim empirically.

In [ ]:
lr_values    = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
SWEEP_EPOCHS = 5
robustness   = {'Adam': {}, 'RAdam': {}}

for lr in lr_values:
    for name, opt_class in [('Adam', optim.Adam), ('RAdam', optim.RAdam)]:
        torch.manual_seed(SEED)
        m    = SimpleCNN(num_classes=NUM_CLASSES).to(device)
        opt  = opt_class(m.parameters(), lr=lr, betas=(BETA1, BETA2), eps=EPS)
        crit = nn.CrossEntropyLoss()
        val_accs = []
        for epoch in range(1, SWEEP_EPOCHS + 1):
            train_one_epoch(m, train_loader, crit, opt, device)
            _, val_acc = evaluate(m, test_loader, crit, device)
            val_accs.append(val_acc)
        robustness[name][lr] = val_accs[-1]
    print(f'lr={lr:.0e} → Adam: {robustness["Adam"][lr]:.2f}% | RAdam: {robustness["RAdam"][lr]:.2f}%')

plt.figure(figsize=(9, 5))
plt.plot(lr_values, list(robustness['Adam'].values()),  marker='o', linewidth=2, label='Adam',  color='tomato')
plt.plot(lr_values, list(robustness['RAdam'].values()), marker='s', linewidth=2, label='RAdam', color='steelblue')
plt.xscale('log')
plt.title(f'LR Robustness — Adam vs RAdam (after {SWEEP_EPOCHS} epochs)', fontsize=12, fontweight='bold')
plt.xlabel('Learning Rate (log scale)'); plt.ylabel('Val Accuracy (%)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/RAdam_lr_robustness.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 11. Cumulative Leaderboard — All 7 Optimizers

In [ ]:
logs_order = [
    ('Vanilla SGD',    'SGD_baseline_log.csv'),
    ('SGD + Momentum', 'SGD_Momentum_log.csv'),
    ('Adagrad',        'Adagrad_log.csv'),
    ('RMSprop',        'RMSprop_log.csv'),
    ('Adam',           'Adam_log.csv'),
    ('AdamW',          'AdamW_log.csv'),
    ('RAdam',          'RAdam_log.csv'),
]

print(f"{'Optimizer':<20} {'Best Val Acc':>14} {'Best Epoch':>11} {'Avg Time/Epoch':>16}")
print('-' * 65)
rows = []
for label, fname in logs_order:
    path = f'{RESULTS_DIR}/{fname}'
    if os.path.exists(path):
        df   = pd.read_csv(path)
        best = df.loc[df['val_acc'].idxmax()]
        rows.append((label, best['val_acc'], int(best['epoch']), df['time_s'].mean()))
        print(f"{label:<20} {best['val_acc']:>13.2f}% {int(best['epoch']):>11} {df['time_s'].mean():>14.1f}s")
    else:
        print(f"{label:<20} {'(not run yet)':>14}")
print('-' * 65)

if rows:
    best_overall = max(rows, key=lambda x: x[1])
    print(f"\n🏆 Best so far: {best_overall[0]} @ {best_overall[1]:.2f}%")

---
## 12. Results Summary & Analysis

In [ ]:
best_epoch = history.loc[history['val_acc'].idxmax()]

print('=' * 55)
print(f'  {OPTIMIZER_NAME} — Summary')
print('=' * 55)
print(f"  Best Val Accuracy : {best_epoch['val_acc']:.2f}%")
print(f"  Best Val Loss     : {best_epoch['val_loss']:.4f}")
print(f"  Achieved at Epoch : {int(best_epoch['epoch'])}")
print(f"  Avg Time/Epoch    : {history['time_s'].mean():.1f}s")
print('=' * 55)
print()
print('📌 Key Observations:')
print('  ✅ More stable early training than Adam — no erratic first epoch')
print('  ✅ Less sensitive to learning rate choice')
print('  ✅ No warmup schedule needed — built-in variance rectification')
print('  ✅ Converges to similar final accuracy as Adam')
print('  ⚠️  Slightly slower than Adam in per-step compute (rectification overhead)')
print('  ⚠️  Less commonly used than Adam/AdamW in practice today')
print()
print('📌 When to use RAdam:')
print('  → Training from scratch without a LR warmup schedule')
print('  → When Adam gives unstable early loss')
print('  → When you want fewer hyperparameters to tune')
print('  → As a reliable baseline for new architectures')

---
## 13. What's Next? — The Final Chapter

All 7 optimizers have now been trained on the same model, same dataset, same seed.  
The final notebook brings everything together:

- 📊 Full head-to-head comparison charts (loss, accuracy, convergence speed)
- 🏆 Final ranked leaderboard with statistical summary
- 📉 Train vs val gap analysis (generalisation comparison)
- ⏱️ Compute efficiency chart (accuracy per second)
- 🗺️ Decision flowchart — which optimizer to use when
- 📋 Cheat sheet table for the report

➡️ Continue to `07_comparison_all.ipynb`